# TE02_020 - Live Collision Avoidance Action Validation

This notebook validates TE02_020 in normal execution mode. It sends sequential goals to `/action/MoveToPosition`, injects a virtual obstacle into the MoveIt planning scene while the real controller is executing, and monitors the controller/action feedback.

The notebook does **not** request plans directly. The real controller must detect collision risk, publish zero velocity as the first safety action, replan automatically, replace the trajectory, and complete the original action goal.

## Expected System Behavior

For each trial:

1. Send goal A or B to `/action/MoveToPosition`.
2. Wait for goal acceptance and execution to start.
3. Publish a virtual obstacle to the planning scene.
4. Monitor feedback and diagnostic events for collision prediction, zero-velocity safety stop, replanning, and trajectory replacement.
5. Wait for the action result after the replanned path completes.
6. Clear the obstacle.
7. Switch to the other goal and repeat.

For strongest timing evidence, the controller publishes JSON events on `/te02_020/collision_avoidance_event`, including `collision_predicted`, `robot_stopped_for_collision`, `replan_started`, `replan_succeeded`, and `trajectory_replaced`. The KPI response time is measured from `collision_predicted` to `robot_stopped_for_collision`.

In [ ]:
from pathlib import Path
import csv
import json
import math
import re
import statistics
import time
import os

import rclpy
from rclpy.action import ActionClient
from rclpy.node import Node
from geometry_msgs.msg import Pose, PoseStamped
from moveit_msgs.msg import CollisionObject, PlanningScene
from moveit_msgs.srv import ApplyPlanningScene
from shape_msgs.msg import SolidPrimitive
from std_msgs.msg import Header, String
from visualization_msgs.msg import Marker, MarkerArray

try:
    from telehandler_moveit.action import MoveToPosition
    ACTION_TYPE_SOURCE = 'telehandler_moveit.action.MoveToPosition'
except Exception:
    from criarte_msgs.action import MoveToPosition
    ACTION_TYPE_SOURCE = 'criarte_msgs.action.MoveToPosition'

KPI_DIR = Path(str(os.environ.get('HOME')) + '/mnt/telehandler/criarte_ws/scripts/Fortis_KPI/KPI/TE02_020')
KPI_DIR.mkdir(parents=True, exist_ok=True)

ACTION_NAME = '/action/MoveToPosition'
APPLY_PLANNING_SCENE_SERVICE = '/apply_planning_scene'
COLLISION_EVENT_TOPIC = '/te02_020/collision_avoidance_event'
TRIAL_EVENT_TOPIC = '/te02_020/trial_event'
TRIAL_RESULT_TOPIC = '/te02_020/trial_result'
MARKER_TOPIC = '/te02_020/markers'

TRIALS_CSV = KPI_DIR / 'te02_020_live_trial_results.csv'
FEEDBACK_CSV = KPI_DIR / 'te02_020_live_feedback.csv'
EVENTS_CSV = KPI_DIR / 'te02_020_live_events.csv'
SUMMARY_CSV = KPI_DIR / 'te02_020_live_summary.csv'

NUM_TRIALS = 10
# Obstacle injection is feedback-triggered. With the common feedback string
# 'Executing trajectory point X/Y', INJECT_WHEN_EXECUTING_POINT=1 injects while point 1
# is active, i.e. before the controller advances to the next trajectory point.
INJECT_WHEN_EXECUTING_POINT = 5
INJECT_WHEN_PROGRESS_GE = 5  # optional float fallback, e.g. 0.10 or 10.0 depending on feedback scale
INJECT_TRIGGER_TIMEOUT_S = 10.0
POST_RESULT_HOLD_S = 1.0
ACTION_TIMEOUT_S = 250.0
SERVER_TIMEOUT_S = 10.0
RESPONSE_THRESHOLD_MS = 250.0
SUCCESS_THRESHOLD_PCT = 95.0

# Use predefined SRDF/controller poses by default. If these are not valid for your current controller,
# set predefined_pose to '' and configure explicit pose fields below.
GOAL_A = {
    'label': 'point_A',
    'predefined_pose': '',
    'frame_id': 'base_link',
    'position': (5.0, 0.0, 1.5),
    'orientation': (0.0, 0.0, 0.0, 1.0),
    'max_velocity': 1.0,
    'factor_scale': 1.0,
}
GOAL_B = {
    'label': 'point_B',
    'predefined_pose': '',
    'frame_id': 'base_link',
    'position': (10.5, 0.0, 1.5),
    'orientation': (0.0, 0.0, 0.0, 1.0),
    'max_velocity': 1.0,
    'factor_scale': 1.0,
}
GOALS = [GOAL_A, GOAL_B]

OBSTACLE = {
    'name': 'te02_020_live_obstacle',
    'frame_id': 'base_link',
    'size_xyz': (0.4, 10.0, 3.0),
    'position_xyz': (8.5, 0.0, 1.0),
}
OBSTACLE2 = {
    'name': 'left_wall',
    'frame_id': 'base_link',
    'size_xyz': (20.0, 0.4, 10.0),
    'position_xyz': (10.0, 3.5, 5.0),
}
OBSTACLE3 = {
    'name': 'right_wall',
    'frame_id': 'base_link',
    'size_xyz': (20.0, 0.4, 10.0),
    'position_xyz': (10.0, -4.0, 5.0),
}
OBSTACLES = [OBSTACLE, OBSTACLE2, OBSTACLE3]

print(f'Action type source: {ACTION_TYPE_SOURCE}')
print(f'Output directory: {KPI_DIR}')

Action type source: telehandler_moveit.action.MoveToPosition
Output directory: /home/lucaspinacosta/mnt/telehandler/criarte_ws/scripts/Fortis_KPI/KPI/TE02_020


In [2]:
def write_csv(path, rows, fieldnames):
    with path.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

def percentile(values, pct):
    if not values:
        return math.nan
    ordered = sorted(values)
    if len(ordered) == 1:
        return ordered[0]
    rank = (pct / 100.0) * (len(ordered) - 1)
    lower = math.floor(rank)
    upper = math.ceil(rank)
    if lower == upper:
        return ordered[int(rank)]
    fraction = rank - lower
    return ordered[lower] * (1.0 - fraction) + ordered[upper] * fraction

def now_ns():
    return time.time_ns()

def monotonic_ms_since(start_ns):
    return (time.monotonic_ns() - start_ns) / 1e6

def make_pose_stamped(goal_cfg):
    msg = PoseStamped()
    msg.header.frame_id = goal_cfg['frame_id']
    msg.pose.position.x, msg.pose.position.y, msg.pose.position.z = goal_cfg['position']
    msg.pose.orientation.x, msg.pose.orientation.y, msg.pose.orientation.z, msg.pose.orientation.w = goal_cfg['orientation']
    return msg

def make_goal_msg(goal_cfg):
    goal = MoveToPosition.Goal()
    goal.target_pose = make_pose_stamped(goal_cfg)
    goal.predefined_pose = goal_cfg['predefined_pose']
    goal.max_velocity = float(goal_cfg['max_velocity'])
    goal.factor_scale = float(goal_cfg['factor_scale'])
    return goal

In [3]:
class TE02020LiveNode(Node):
    def __init__(self):
        super().__init__('te02_020_live_action_collision_validation')
        self.action_client = ActionClient(self, MoveToPosition, ACTION_NAME)
        self.apply_scene_client = self.create_client(ApplyPlanningScene, APPLY_PLANNING_SCENE_SERVICE)
        self.trial_event_pub = self.create_publisher(String, TRIAL_EVENT_TOPIC, 10)
        self.trial_result_pub = self.create_publisher(String, TRIAL_RESULT_TOPIC, 10)
        self.marker_pub = self.create_publisher(MarkerArray, MARKER_TOPIC, 10)
        self.create_subscription(String, COLLISION_EVENT_TOPIC, self.on_collision_event, 50)
        self.feedback_rows = []
        self.event_rows = []
        self.current_trial_id = None
        self.current_obstacle_injection_time_ns = None
        self.detected_event_time_ns = None
        self.robot_stopped_event_time_ns = None
        self.hit_event_time_ns = None
        self.replan_event_time_ns = None
        self.replan_succeeded_time_ns = None
        self.trajectory_replaced_time_ns = None
        self.injection_triggered = False
        self.injection_trigger_time_ns = None
        self.injection_trigger_reason = ''

    def wait_ready(self):
        action_ready = self.action_client.wait_for_server(timeout_sec=SERVER_TIMEOUT_S)
        scene_ready = self.apply_scene_client.wait_for_service(timeout_sec=SERVER_TIMEOUT_S)
        return action_ready and scene_ready

    def publish_json(self, pub, payload):
        pub.publish(String(data=json.dumps(payload, sort_keys=True)))

    def on_collision_event(self, msg):
        received_ns = now_ns()
        try:
            payload = json.loads(msg.data)
        except json.JSONDecodeError:
            payload = {'event': msg.data}
        event = str(payload.get('event', '')).lower()
        row = {
            'trial_id': self.current_trial_id,
            'receive_time_ns': received_ns,
            'event': event,
            'raw_data': msg.data,
        }
        self.event_rows.append(row)
        if self.current_obstacle_injection_time_ns is not None:
            if self.detected_event_time_ns is None and ('collision' in event or 'avoid' in event):
                self.detected_event_time_ns = received_ns
            if self.robot_stopped_event_time_ns is None and event == 'robot_stopped_for_collision':
                self.robot_stopped_event_time_ns = received_ns
            if self.hit_event_time_ns is None and any(k in event for k in ('hit', 'impact', 'contact')):
                self.hit_event_time_ns = received_ns
            if self.replan_event_time_ns is None and 'replan' in event:
                self.replan_event_time_ns = received_ns
            if self.replan_succeeded_time_ns is None and event == 'replan_succeeded':
                self.replan_succeeded_time_ns = received_ns
            if self.trajectory_replaced_time_ns is None and event == 'trajectory_replaced':
                self.trajectory_replaced_time_ns = received_ns

    def maybe_trigger_injection_from_feedback(self, received_ns, progress, status):
        if self.injection_triggered:
            return
        status_l = status.lower()
        point_match = re.search(r'point\s+(\d+)\s*/\s*(\d+)', status_l)
        if point_match:
            current_point = int(point_match.group(1))
            total_points = int(point_match.group(2))
            if current_point >= total_points / INJECT_WHEN_EXECUTING_POINT:
                self.injection_triggered = True
                self.injection_trigger_time_ns = received_ns
                self.injection_trigger_reason = f'feedback_point_{current_point}_of_{total_points}'
                return
        if INJECT_WHEN_PROGRESS_GE is not None and progress >= float(INJECT_WHEN_PROGRESS_GE):
            self.injection_triggered = True
            self.injection_trigger_time_ns = received_ns
            self.injection_trigger_reason = f'feedback_progress_{progress}'

    def feedback_callback(self, feedback_msg):
        received_ns = now_ns()
        feedback = feedback_msg.feedback
        status = str(getattr(feedback, 'status', ''))
        progress = float(getattr(feedback, 'feedback', 0.0))
        self.feedback_rows.append({
            'trial_id': self.current_trial_id,
            'receive_time_ns': received_ns,
            'feedback': progress,
            'status': status,
        })
        self.maybe_trigger_injection_from_feedback(received_ns, progress, status)
        status_l = status.lower()
        if self.current_obstacle_injection_time_ns is not None:
            if self.detected_event_time_ns is None and ('collision' in status_l or 'avoid' in status_l):
                self.detected_event_time_ns = received_ns
            if self.hit_event_time_ns is None and any(k in status_l for k in ('hit', 'impact', 'contact')):
                self.hit_event_time_ns = received_ns
            if self.replan_event_time_ns is None and 'replan' in status_l:
                self.replan_event_time_ns = received_ns

    def make_obstacle_object(self, operation, obstacle_cfg=OBSTACLE):
        obj = CollisionObject()
        obj.header.frame_id = obstacle_cfg['frame_id']
        obj.id = obstacle_cfg['name']
        obj.operation = operation
        if operation == CollisionObject.REMOVE:
            return obj
        primitive = SolidPrimitive()
        primitive.type = SolidPrimitive.BOX
        primitive.dimensions = [float(v) for v in obstacle_cfg['size_xyz']]
        pose = Pose()
        pose.position.x, pose.position.y, pose.position.z = obstacle_cfg['position_xyz']
        pose.orientation.w = 1.0
        obj.primitives.append(primitive)
        obj.primitive_poses.append(pose)
        return obj
    

    def apply_obstacle(self, operation):
        scene = PlanningScene()
        scene.is_diff = True
        scene.world.collision_objects = [
            self.make_obstacle_object(operation, obstacle_cfg)
            for obstacle_cfg in OBSTACLES
        ]
        req = ApplyPlanningScene.Request()
        req.scene = scene
        start_ns = time.monotonic_ns()
        future = self.apply_scene_client.call_async(req)
        deadline = time.monotonic() + SERVER_TIMEOUT_S
        while rclpy.ok() and not future.done() and time.monotonic() < deadline:
            rclpy.spin_once(self, timeout_sec=0.01)
        elapsed_ms = monotonic_ms_since(start_ns)
        if not future.done():
            return False, elapsed_ms
        result = future.result()
        return bool(result and result.success), elapsed_ms

    def clear_markers(self):
        markers = MarkerArray()
        stamp = self.get_clock().now().to_msg()
        for marker_id, obstacle_cfg in enumerate(OBSTACLES, start=1):
            marker = Marker()
            marker.header.frame_id = obstacle_cfg['frame_id']
            marker.header.stamp = stamp
            marker.ns = 'te02_020_live_obstacle'
            marker.id = marker_id
            marker.action = Marker.DELETE
            markers.markers.append(marker)
        for marker_id in (2, 100):
            marker = Marker()
            marker.header.frame_id = OBSTACLES[-1]['frame_id']
            marker.header.stamp = stamp
            marker.ns = 'te02_020_live_status'
            marker.id = marker_id
            marker.action = Marker.DELETE
            markers.markers.append(marker)
        self.marker_pub.publish(markers)

    def publish_marker(self, text, success=None, show_obstacle=True):
        markers = MarkerArray()
        stamp = self.get_clock().now().to_msg()
        for marker_id, obstacle_cfg in enumerate(OBSTACLES, start=1):
            cube = Marker()
            cube.header.frame_id = obstacle_cfg['frame_id']
            cube.header.stamp = stamp
            cube.ns = 'te02_020_live_obstacle'
            cube.id = marker_id
            cube.type = Marker.CUBE
            cube.action = Marker.ADD if show_obstacle else Marker.DELETE
            cube.pose.position.x, cube.pose.position.y, cube.pose.position.z = obstacle_cfg['position_xyz']
            cube.pose.orientation.w = 1.0
            cube.scale.x, cube.scale.y, cube.scale.z = obstacle_cfg['size_xyz']
            cube.color.r = 1.0
            cube.color.g = 0.25
            cube.color.b = 0.05
            cube.color.a = 0.45
            markers.markers.append(cube)
        label_cfg = OBSTACLES[-1]
        label = Marker()
        label.header.frame_id = label_cfg['frame_id']
        label.header.stamp = stamp
        label.ns = 'te02_020_live_status'
        label.id = 100
        label.type = Marker.TEXT_VIEW_FACING
        label.action = Marker.ADD
        label.pose.position.x, label.pose.position.y, label.pose.position.z = label_cfg['position_xyz']
        label.pose.position.z += max(label_cfg['size_xyz'][2], 0.5) * 0.75
        label.pose.orientation.w = 1.0
        label.scale.z = 0.25
        label.text = text
        if success is None:
            label.color.r = label.color.g = label.color.b = 1.0
        elif success:
            label.color.g = 1.0
        else:
            label.color.r = 1.0
        label.color.a = 1.0
        markers.markers.append(label)
        self.marker_pub.publish(markers)

In [4]:
def run_trial(node, trial_id, goal_cfg):
    node.current_trial_id = trial_id
    node.current_obstacle_injection_time_ns = None
    node.detected_event_time_ns = None
    node.robot_stopped_event_time_ns = None
    node.hit_event_time_ns = None
    node.replan_event_time_ns = None
    node.replan_succeeded_time_ns = None
    node.trajectory_replaced_time_ns = None
    node.injection_triggered = False
    node.injection_trigger_time_ns = None
    node.injection_trigger_reason = ''

    preclear_ok, preclear_ms = node.apply_obstacle(CollisionObject.REMOVE)
    node.clear_markers()
    node.publish_json(node.trial_event_pub, {
        'trial_id': trial_id,
        'event': 'obstacle_removed_before_trial',
        'time_ns': now_ns(),
        'apply_success': preclear_ok,
        'apply_ms': preclear_ms,
    })
    node.publish_json(node.trial_event_pub, {
        'trial_id': trial_id,
        'event': 'goal_send',
        'goal_label': goal_cfg['label'],
        'predefined_pose': goal_cfg['predefined_pose'],
        'time_ns': now_ns(),
    })

    goal_msg = make_goal_msg(goal_cfg)
    send_start_ns = time.monotonic_ns()
    send_future = node.action_client.send_goal_async(goal_msg, feedback_callback=node.feedback_callback)
    deadline = time.monotonic() + SERVER_TIMEOUT_S
    while rclpy.ok() and not send_future.done() and time.monotonic() < deadline:
        rclpy.spin_once(node, timeout_sec=0.01)
    if not send_future.done():
        return {'trial_id': trial_id, 'goal_label': goal_cfg['label'], 'status': 'FAIL', 'failure_reason': 'goal_send_timeout'}

    goal_handle = send_future.result()
    goal_accepted = bool(goal_handle and goal_handle.accepted)
    goal_accept_ms = monotonic_ms_since(send_start_ns)
    if not goal_accepted:
        return {'trial_id': trial_id, 'goal_label': goal_cfg['label'], 'goal_accepted': False, 'status': 'FAIL', 'failure_reason': 'goal_rejected'}

    node.publish_json(node.trial_event_pub, {'trial_id': trial_id, 'event': 'goal_accepted', 'time_ns': now_ns()})
    node.publish_marker(f"TE02_020 live trial {trial_id}\nGoal {goal_cfg['label']} accepted\nWaiting for collision-test trigger", show_obstacle=False)

    trigger_deadline = time.monotonic() + INJECT_TRIGGER_TIMEOUT_S
    while rclpy.ok() and not node.injection_triggered and time.monotonic() < trigger_deadline:
        rclpy.spin_once(node, timeout_sec=0.02)
    if not node.injection_triggered:
        node.injection_triggered = True
        node.injection_trigger_time_ns = now_ns()
        node.injection_trigger_reason = 'trigger_timeout_fallback'

    obstacle_injection_wall_ns = now_ns()
    node.current_obstacle_injection_time_ns = obstacle_injection_wall_ns
    obstacle_ok, obstacle_apply_ms = node.apply_obstacle(CollisionObject.ADD)
    node.publish_json(node.trial_event_pub, {
        'trial_id': trial_id,
        'event': 'obstacle_injected',
        'time_ns': obstacle_injection_wall_ns,
        'apply_success': obstacle_ok,
        'apply_ms': obstacle_apply_ms,
        'trigger_reason': node.injection_trigger_reason,
        'trigger_time_ns': node.injection_trigger_time_ns,
        'obstacles': OBSTACLES,
    })
    node.publish_marker(f"TE02_020 live trial {trial_id}\nObstacle injected\n{node.injection_trigger_reason}")

    result_future = goal_handle.get_result_async()
    action_deadline = time.monotonic() + ACTION_TIMEOUT_S
    while rclpy.ok() and not result_future.done() and time.monotonic() < action_deadline:
        rclpy.spin_once(node, timeout_sec=0.05)

    result_received = result_future.done()
    action_success = False
    action_result_text = ''
    if result_received:
        result = result_future.result().result
        action_success = bool(getattr(result, 'success', False))
        action_result_text = str(getattr(result, 'result', ''))

    obstacle_to_detection_ms = ''
    detection_to_stop_ms = ''
    replan_response_ms = ''
    replan_succeeded_ms = ''
    trajectory_replaced_ms = ''
    if node.detected_event_time_ns is not None:
        obstacle_to_detection_ms = (node.detected_event_time_ns - obstacle_injection_wall_ns) / 1e6
    if node.detected_event_time_ns is not None and node.robot_stopped_event_time_ns is not None:
        detection_to_stop_ms = (node.robot_stopped_event_time_ns - node.detected_event_time_ns) / 1e6
    if node.replan_event_time_ns is not None:
        replan_response_ms = (node.replan_event_time_ns - obstacle_injection_wall_ns) / 1e6
    if node.replan_succeeded_time_ns is not None and node.detected_event_time_ns is not None:
        replan_succeeded_ms = (node.replan_succeeded_time_ns - node.detected_event_time_ns) / 1e6
    if node.trajectory_replaced_time_ns is not None and node.detected_event_time_ns is not None:
        trajectory_replaced_ms = (node.trajectory_replaced_time_ns - node.detected_event_time_ns) / 1e6
    hit_before_stop = bool(
        node.hit_event_time_ns is not None and
        (node.robot_stopped_event_time_ns is None or node.hit_event_time_ns <= node.robot_stopped_event_time_ns)
    )

    if node.detected_event_time_ns is None:
        timing_pass = False
        timing_reason = 'no_collision_predicted_event_observed'
    elif detection_to_stop_ms == '':
        timing_pass = False
        timing_reason = 'no_robot_stopped_for_collision_event_observed'
    else:
        timing_pass = float(detection_to_stop_ms) < RESPONSE_THRESHOLD_MS
        timing_reason = 'none' if timing_pass else 'response_time_over_threshold'

    replan_success = bool(node.replan_succeeded_time_ns is not None and node.trajectory_replaced_time_ns is not None)
    # TE02_020 success requires an immediate safety stop, successful replan,
    # trajectory replacement, and completion of the original action goal.
    trial_success = bool(obstacle_ok and timing_pass and replan_success and result_received and action_success and not hit_before_stop)
    failure_reasons = []
    if not obstacle_ok:
        failure_reasons.append('obstacle_apply_failed')
    if timing_reason != 'none':
        failure_reasons.append(timing_reason)
    if not replan_success:
        failure_reasons.append('replan_or_trajectory_replacement_not_observed')
    if not result_received:
        failure_reasons.append('action_result_timeout')
    elif not action_success:
        failure_reasons.append('action_result_failed')
    if hit_before_stop:
        failure_reasons.append('hit_or_contact_before_stop')
    if not failure_reasons:
        failure_reasons.append('none')

    node.publish_marker(
        f"TE02_020 live trial {trial_id}: {'PASS' if trial_success else 'FAIL'}\ndetection_to_stop={detection_to_stop_ms} ms",
        success=trial_success,
    )
    node.publish_json(node.trial_result_pub, {'trial_id': trial_id, 'success': trial_success, 'failure_reason': ';'.join(failure_reasons)})
    time.sleep(POST_RESULT_HOLD_S)
    obstacle_remove_ok, obstacle_remove_ms = node.apply_obstacle(CollisionObject.REMOVE)
    node.clear_markers()
    node.publish_json(node.trial_event_pub, {
        'trial_id': trial_id,
        'event': 'obstacle_removed_after_action_complete',
        'time_ns': now_ns(),
        'apply_success': obstacle_remove_ok,
        'apply_ms': obstacle_remove_ms,
    })

    return {
        'trial_id': trial_id,
        'goal_label': goal_cfg['label'],
        'predefined_pose': goal_cfg['predefined_pose'],
        'goal_accepted': goal_accepted,
        'goal_accept_ms': round(goal_accept_ms, 3),
        'obstacle_apply_success': obstacle_ok,
        'obstacle_apply_ms': round(obstacle_apply_ms, 3),
        'obstacle_remove_success': obstacle_remove_ok,
        'obstacle_remove_ms': round(obstacle_remove_ms, 3),
        'injection_trigger_reason': node.injection_trigger_reason,
        'injection_trigger_time_ns': node.injection_trigger_time_ns,
        'result_received': result_received,
        'action_success': action_success,
        'action_result_text': action_result_text,
        'obstacle_to_detection_ms': round(obstacle_to_detection_ms, 3) if obstacle_to_detection_ms != '' else '',
        'detection_to_stop_ms': round(detection_to_stop_ms, 3) if detection_to_stop_ms != '' else '',
        'replan_response_ms': round(replan_response_ms, 3) if replan_response_ms != '' else '',
        'replan_succeeded_ms': round(replan_succeeded_ms, 3) if replan_succeeded_ms != '' else '',
        'trajectory_replaced_ms': round(trajectory_replaced_ms, 3) if trajectory_replaced_ms != '' else '',
        'replan_success': replan_success,
        'hit_before_stop': hit_before_stop,
        'success': trial_success,
        'failure_reason': ';'.join(failure_reasons),
    }

## Run Sequential Alternating Trials

Start RViz and bag recording before running this cell. The goal alternates A/B after each action result.

In [5]:
if not rclpy.ok():
    rclpy.init()
node = TE02020LiveNode()
if not node.wait_ready():
    raise RuntimeError('Action server or /apply_planning_scene service is not ready')

trial_rows = []
try:
    for trial_id in range(1, NUM_TRIALS + 1):
        goal_cfg = GOALS[(trial_id - 1) % len(GOALS)]
        row = run_trial(node, trial_id, goal_cfg)
        trial_rows.append(row)
        print(f"trial={trial_id}/{NUM_TRIALS} goal={goal_cfg['label']} success={row.get('success')} reason={row.get('failure_reason')}")
finally:
    node.apply_obstacle(CollisionObject.REMOVE)
    node.clear_markers()

print(f'Completed {len(trial_rows)} trials')

trial=1/10 goal=point_A success=False reason=no_collision_predicted_event_observed;replan_or_trajectory_replacement_not_observed
trial=2/10 goal=point_B success=False reason=action_result_failed
trial=3/10 goal=point_A success=False reason=no_collision_predicted_event_observed;replan_or_trajectory_replacement_not_observed
trial=4/10 goal=point_B success=True reason=none
trial=5/10 goal=point_A success=True reason=none
trial=6/10 goal=point_B success=True reason=none
trial=7/10 goal=point_A success=True reason=none
trial=8/10 goal=point_B success=True reason=none
trial=9/10 goal=point_A success=True reason=none
trial=10/10 goal=point_B success=True reason=none
Completed 10 trials


In [6]:
trial_fields = [
    'trial_id', 'goal_label', 'predefined_pose', 'goal_accepted', 'goal_accept_ms',
    'obstacle_apply_success', 'obstacle_apply_ms', 'obstacle_remove_success', 'obstacle_remove_ms',
    'injection_trigger_reason', 'injection_trigger_time_ns',
    'result_received', 'action_success',
    'action_result_text', 'obstacle_to_detection_ms', 'detection_to_stop_ms', 'replan_response_ms',
    'replan_succeeded_ms', 'trajectory_replaced_ms', 'replan_success',
    'hit_before_stop', 'success', 'failure_reason'
]
feedback_fields = ['trial_id', 'receive_time_ns', 'feedback', 'status']
event_fields = ['trial_id', 'receive_time_ns', 'event', 'raw_data']
write_csv(TRIALS_CSV, trial_rows, trial_fields)
write_csv(FEEDBACK_CSV, node.feedback_rows, feedback_fields)
write_csv(EVENTS_CSV, node.event_rows, event_fields)

total = len(trial_rows)
successes = sum(1 for row in trial_rows if row.get('success') is True)
success_rate = successes / total * 100.0 if total else 0.0
latencies = [float(row['detection_to_stop_ms']) for row in trial_rows if row.get('detection_to_stop_ms') != '']
obstacle_to_detection_latencies = [float(row['obstacle_to_detection_ms']) for row in trial_rows if row.get('obstacle_to_detection_ms') != '']
max_latency = max(latencies) if latencies else math.nan
p95_latency = percentile(latencies, 95) if latencies else math.nan
obstacle_to_detection_max = max(obstacle_to_detection_latencies) if obstacle_to_detection_latencies else math.nan
obstacle_to_detection_p95 = percentile(obstacle_to_detection_latencies, 95) if obstacle_to_detection_latencies else math.nan
over_threshold = sum(1 for value in latencies if value >= RESPONSE_THRESHOLD_MS)
hit_before_stop_trials = sum(1 for row in trial_rows if row.get('hit_before_stop') is True)
replan_successes = sum(1 for row in trial_rows if row.get('replan_success') is True)
action_successes = sum(1 for row in trial_rows if row.get('action_success') is True)
overall = 'PASS' if total and success_rate > SUCCESS_THRESHOLD_PCT and latencies and max_latency < RESPONSE_THRESHOLD_MS else 'FAIL'
summary_rows = [
    {'metric': 'action_name', 'value': ACTION_NAME, 'status': 'INFO', 'details': ACTION_TYPE_SOURCE},
    {'metric': 'total_trials', 'value': total, 'status': 'INFO', 'details': ''},
    {'metric': 'successful_trials', 'value': successes, 'status': 'PASS' if success_rate > SUCCESS_THRESHOLD_PCT else 'FAIL', 'details': f'threshold>{SUCCESS_THRESHOLD_PCT}%'},
    {'metric': 'avoidance_success_rate_pct', 'value': round(success_rate, 3), 'status': 'PASS' if success_rate > SUCCESS_THRESHOLD_PCT else 'FAIL', 'details': f'threshold>{SUCCESS_THRESHOLD_PCT}%'},
    {'metric': 'response_threshold_ms', 'value': RESPONSE_THRESHOLD_MS, 'status': 'INFO', 'details': ''},
    {'metric': 'response_time_definition', 'value': 'collision_predicted_to_robot_stopped_for_collision', 'status': 'INFO', 'details': 'controller response from detection to first safety action: zero velocity publication'},
    {'metric': 'obstacle_to_detection_samples', 'value': len(obstacle_to_detection_latencies), 'status': 'INFO', 'details': 'diagnostic only: obstacle injection to collision_predicted'},
    {'metric': 'obstacle_to_detection_p95_ms', 'value': round(obstacle_to_detection_p95, 3) if obstacle_to_detection_latencies else '', 'status': 'INFO', 'details': 'diagnostic only'},
    {'metric': 'obstacle_to_detection_max_ms', 'value': round(obstacle_to_detection_max, 3) if obstacle_to_detection_latencies else '', 'status': 'INFO', 'details': 'diagnostic only'},
    {'metric': 'detection_to_stop_samples', 'value': len(latencies), 'status': 'PASS' if len(latencies) == total and total else 'FAIL', 'details': 'requires collision_predicted and robot_stopped_for_collision events'},
    {'metric': 'detection_to_stop_p95_ms', 'value': round(p95_latency, 3) if latencies else '', 'status': 'PASS' if latencies and p95_latency < RESPONSE_THRESHOLD_MS else 'FAIL', 'details': f'threshold<{RESPONSE_THRESHOLD_MS} ms'},
    {'metric': 'detection_to_stop_max_ms', 'value': round(max_latency, 3) if latencies else '', 'status': 'PASS' if latencies and max_latency < RESPONSE_THRESHOLD_MS else 'FAIL', 'details': f'threshold<{RESPONSE_THRESHOLD_MS} ms'},
    {'metric': 'replan_successful_trials', 'value': replan_successes, 'status': 'PASS' if replan_successes == total and total else 'FAIL', 'details': 'requires replan_succeeded and trajectory_replaced events'},
    {'metric': 'action_successful_trials', 'value': action_successes, 'status': 'PASS' if action_successes == total and total else 'FAIL', 'details': 'action completes after avoidance'},
    {'metric': 'hit_before_stop_trials', 'value': hit_before_stop_trials, 'status': 'PASS' if hit_before_stop_trials == 0 and total else 'FAIL', 'details': 'from optional hit/contact/impact diagnostics'},
    {'metric': 'response_over_threshold_trials', 'value': over_threshold, 'status': 'PASS' if over_threshold == 0 and latencies else 'FAIL', 'details': ''},
    {'metric': 'TE02_020_overall', 'value': overall, 'status': overall, 'details': f'{successes}/{total} trials successful'},
]
write_csv(SUMMARY_CSV, summary_rows, ['metric', 'value', 'status', 'details'])
print(f'Wrote {TRIALS_CSV}')
print(f'Wrote {FEEDBACK_CSV}')
print(f'Wrote {EVENTS_CSV}')
print(f'Wrote {SUMMARY_CSV}')
for row in summary_rows:
    print(f"{row['metric']}: {row['value']} [{row['status']}] {row['details']}")

Wrote /home/lucaspinacosta/mnt/telehandler/criarte_ws/scripts/Fortis_KPI/KPI/TE02_020/te02_020_live_trial_results.csv
Wrote /home/lucaspinacosta/mnt/telehandler/criarte_ws/scripts/Fortis_KPI/KPI/TE02_020/te02_020_live_feedback.csv
Wrote /home/lucaspinacosta/mnt/telehandler/criarte_ws/scripts/Fortis_KPI/KPI/TE02_020/te02_020_live_events.csv
Wrote /home/lucaspinacosta/mnt/telehandler/criarte_ws/scripts/Fortis_KPI/KPI/TE02_020/te02_020_live_summary.csv
action_name: /action/MoveToPosition [INFO] telehandler_moveit.action.MoveToPosition
total_trials: 10 [INFO] 
successful_trials: 7 [FAIL] threshold>95.0%
avoidance_success_rate_pct: 70.0 [FAIL] threshold>95.0%
response_threshold_ms: 250.0 [INFO] 
response_time_definition: collision_predicted_to_robot_stopped_for_collision [INFO] controller response from detection to first safety action: zero velocity publication
obstacle_to_detection_samples: 8 [INFO] diagnostic only: obstacle injection to collision_predicted
obstacle_to_detection_p95_ms: 33

In [7]:
node.destroy_node()
rclpy.shutdown()